# 🚀 Model Export and Deployment

This notebook exports the trained LoRA and OSFT models and deploys them to RHOAI 3.5.

## Processing Steps

| # | Step | Description |
|---|------|------|
| 1 | LoRA export | Save adapter + merge with base model (merged model) |
| 2 | OSFT export | Convert to HuggingFace format |
| 3 | Validation | Reload test, hash check |
| 4 | S3 upload | Transfer to S3-compatible storage |
| 5 | Serving manifests | Generate and apply InferenceService YAML |
| 6 | Smoke test | Chat completion + tool call test |

### Prerequisites
- `03_lora_finetuning.ipynb` or `04_osft_finetuning.ipynb` completed
- S3 access configured (for upload)
- RHOAI cluster access (for deployment)

In [ ]:
"""Export LoRA adapter and merge to full model."""

import os
import json
import hashlib
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_training_config, PROJECT_ROOT,
)

load_env()

lora_config = load_training_config("lora")
model_id = lora_config["model"]["model_id"]

adapter_path = PROJECT_ROOT / lora_config["export"]["adapter_path"]
merged_path = PROJECT_ROOT / lora_config["export"]["merged_path"]
checkpoint_dir = PROJECT_ROOT / lora_config["training_args"]["output_dir"]

print("=" * 70)
print("📦 LoRA Model Export")
print("=" * 70)

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    import torch

    # 1. Save adapter separately
    print("\n1️⃣ Saving LoRA adapter...")
    adapter_path.mkdir(parents=True, exist_ok=True)

    # Copy adapter files from checkpoint
    adapter_source = checkpoint_dir
    adapter_config_file = adapter_source / "adapter_config.json"
    if not adapter_config_file.exists():
        checkpoints = sorted(checkpoint_dir.glob("checkpoint-*"))
        if checkpoints:
            adapter_source = checkpoints[-1]

    import shutil
    for f in adapter_source.glob("adapter_*"):
        shutil.copy2(f, adapter_path / f.name)
    for f in adapter_source.glob("tokenizer*"):
        shutil.copy2(f, adapter_path / f.name)
    if (adapter_source / "special_tokens_map.json").exists():
        shutil.copy2(adapter_source / "special_tokens_map.json", adapter_path)

    print(f"  ✅ Adapter saved: {adapter_path}")

    # 2. Merge adapter with base model
    if lora_config["export"].get("save_merged", True):
        print("\n2️⃣ Merging adapter with base model...")
        print("  (This process may take several minutes)")

        base_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            device_map="auto",
        )
        model = PeftModel.from_pretrained(base_model, str(adapter_source))
        merged_model = model.merge_and_unload()

        merged_path.mkdir(parents=True, exist_ok=True)
        merged_model.save_pretrained(str(merged_path))

        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        tokenizer.save_pretrained(str(merged_path))

        print(f"  ✅ Merged model saved: {merged_path}")
        del base_model, model, merged_model
        torch.cuda.empty_cache()
    else:
        print("  ⏭️ Merge skipped (disabled in config)")

except Exception as exc:
    print(f"❌ LoRA export failed: {exc}")
    print("  If GPU memory is insufficient, run the merge in a separate process.")
    raise

In [ ]:
"""Export OSFT model to HuggingFace format."""

osft_config = load_training_config("osft")
osft_checkpoint = PROJECT_ROOT / osft_config["training_args"]["output_dir"]
osft_export_path = PROJECT_ROOT / osft_config["export"]["output_path"]

print("=" * 70)
print("📦 OSFT Model Export")
print("=" * 70)

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
    import torch

    print(f"\nOSFT checkpoint: {osft_checkpoint}")
    print(f"Export path: {osft_export_path}")

    # OSFT exports the full model (not an adapter)
    print("\nLoading model and converting to HuggingFace format...")
    osft_model = AutoModelForCausalLM.from_pretrained(
        str(osft_checkpoint),
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map="auto",
    )

    osft_export_path.mkdir(parents=True, exist_ok=True)
    osft_model.save_pretrained(str(osft_export_path))

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.save_pretrained(str(osft_export_path))

    print(f"✅ OSFT model export complete: {osft_export_path}")

    del osft_model
    torch.cuda.empty_cache()

except Exception as exc:
    print(f"❌ OSFT export failed: {exc}")
    raise

In [ ]:
"""Validate exported models — reload test and hash check."""

from rhoai_model_training_lab.data import compute_file_checksum

print("=" * 70)
print("🔍 Export Validation")
print("=" * 70)

validation_results = {}

# Validate LoRA merged model
for name, model_path in [("LoRA (merged)", merged_path), ("OSFT", osft_export_path)]:
    print(f"\n--- {name} Validation ---")
    checks = {}

    # Check required files
    required = ["config.json", "tokenizer_config.json"]
    for req in required:
        exists = (model_path / req).exists()
        checks[req] = exists
        print(f"  {'✅' if exists else '❌'} {req}")

    # Check weight files
    safetensors = list(model_path.glob("*.safetensors"))
    bin_files = list(model_path.glob("*.bin"))
    weight_files = safetensors or bin_files
    checks["weights"] = bool(weight_files)
    print(f"  {'✅' if weight_files else '❌'} Weight files: {len(weight_files)}")

    # Check index file
    index = (
        (model_path / "model.safetensors.index.json").exists()
        or (model_path / "pytorch_model.bin.index.json").exists()
        or len(weight_files) == 1
    )
    checks["index"] = index
    print(f"  {'✅' if index else '⚠️ '} Index file")

    # Compute hash of weight files
    if weight_files:
        h = hashlib.sha256()
        for wf in sorted(weight_files):
            h.update(compute_file_checksum(wf).encode())
        model_hash = h.hexdigest()[:16]
        print(f"  📎 Model hash: {model_hash}")

    # Reload test
    try:
        config = AutoConfig.from_pretrained(str(model_path), trust_remote_code=True)
        tok = AutoTokenizer.from_pretrained(str(model_path), trust_remote_code=True)
        checks["reload"] = True
        print(f"  ✅ Reload test passed ({config.model_type})")
    except Exception as exc:
        checks["reload"] = False
        print(f"  ❌ Reload failed: {exc}")

    # Total size
    total_bytes = sum(f.stat().st_size for f in model_path.rglob("*") if f.is_file())
    print(f"  📦 Total size: {total_bytes / (1024**3):.2f} GB")

    validation_results[name] = all(checks.values())

all_valid = all(validation_results.values())
print(f"\n{'✅ All model validations passed' if all_valid else '❌ Some validations failed'}")

In [ ]:
"""Upload exported models to S3."""

s3_endpoint = os.environ.get("S3_ENDPOINT", "")
s3_bucket = os.environ.get("S3_BUCKET", "rhoai-model-training-lab")

print("=" * 70)
print("☁️  S3 Upload")
print("=" * 70)

if not s3_endpoint:
    print("⚠️  S3_ENDPOINT is not set.")
    print("   Skipping S3 upload.")
    print("   Manual upload commands:")
    print(f"   aws s3 sync {merged_path} s3://{s3_bucket}/models/lora-merged/")
    print(f"   aws s3 sync {osft_export_path} s3://{s3_bucket}/models/osft-exported/")
else:
    try:
        import boto3

        s3 = boto3.client(
            "s3",
            endpoint_url=s3_endpoint,
            aws_access_key_id=os.environ.get("S3_ACCESS_KEY_ID"),
            aws_secret_access_key=os.environ.get("S3_SECRET_ACCESS_KEY"),
            region_name=os.environ.get("S3_REGION", "us-east-1"),
        )

        uploads = [
            (merged_path, "models/lora-merged"),
            (osft_export_path, "models/osft-exported"),
            (adapter_path, "models/lora-adapter"),
        ]

        for local_path, s3_prefix in uploads:
            if not local_path.exists():
                print(f"  ⏭️ {local_path.name} not found, skipped")
                continue
            print(f"\n  📤 {local_path.name} → s3://{s3_bucket}/{s3_prefix}/")
            for file_path in local_path.rglob("*"):
                if file_path.is_file():
                    rel = file_path.relative_to(local_path)
                    s3_key = f"{s3_prefix}/{rel}"
                    s3.upload_file(str(file_path), s3_bucket, s3_key)
            print(f"  ✅ Upload complete")

    except ImportError:
        print("❌ boto3 is not installed. pip install boto3")
    except Exception as exc:
        print(f"❌ S3 upload failed: {exc}")

In [ ]:
"""Render and apply serving manifests (InferenceService)."""

from jinja2 import Template

print("=" * 70)
print("📋 Serving Manifest Generation")
print("=" * 70)

INFERENCE_SERVICE_TEMPLATE = """apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: {{ name }}
  namespace: {{ namespace }}
  annotations:
    serving.kserve.io/deploymentMode: RawDeployment
spec:
  predictor:
    model:
      modelFormat:
        name: vLLM
      runtime: vllm-runtime
      storageUri: {{ storage_uri }}
      resources:
        limits:
          nvidia.com/gpu: "1"
        requests:
          cpu: "4"
          memory: "16Gi"
      env:
        - name: MODEL_NAME
          value: "{{ model_name }}"
        - name: MAX_MODEL_LEN
          value: "4096"
        - name: TOOL_PARSER
          value: "{{ tool_parser }}"
"""

template = Template(INFERENCE_SERVICE_TEMPLATE)
namespace = os.environ.get("NAMESPACE", "rhoai-lab")

manifests = [
    {
        "name": "lora-model-predictor",
        "model_name": "Qwen3-4B-Instruct-2507-lora",
        "storage_uri": f"s3://{s3_bucket}/models/lora-merged",
    },
    {
        "name": "osft-model-predictor",
        "model_name": "Qwen3-4B-Instruct-2507-osft",
        "storage_uri": f"s3://{s3_bucket}/models/osft-exported",
    },
]

manifest_dir = PROJECT_ROOT / "deploy" / "manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)

for m in manifests:
    rendered = template.render(
        name=m["name"],
        namespace=namespace,
        storage_uri=m["storage_uri"],
        model_name=m["model_name"],
        tool_parser="hermes",
    )
    manifest_file = manifest_dir / f"{m['name']}.yaml"
    manifest_file.write_text(rendered)
    print(f"\n✅ Manifest generated: {manifest_file}")
    print(rendered)

print("\n📝 Deployment commands:")
for m in manifests:
    print(f"  oc apply -f deploy/manifests/{m['name']}.yaml")
print(f"\n  oc get inferenceservice -n {namespace}")

In [ ]:
"""Smoke test — chat completion and tool call test."""

from rhoai_model_training_lab.config import load_yaml_config

print("=" * 70)
print("🧪 Serving Smoke Test")
print("=" * 70)

# Check configured endpoints
endpoints_config_path = PROJECT_ROOT / "configs" / "endpoints.example.yaml"
try:
    endpoints_config = load_yaml_config(endpoints_config_path)
except FileNotFoundError:
    endpoints_config = {"endpoints": {}}

test_endpoints = {
    "base": os.environ.get("BASE_SERVING_ENDPOINT", ""),
    "lora": os.environ.get("LORA_SERVING_ENDPOINT", ""),
    "osft": os.environ.get("OSFT_SERVING_ENDPOINT", ""),
}

import httpx

for variant, endpoint in test_endpoints.items():
    if not endpoint:
        print(f"\n⏭️ {variant}: endpoint not set, skipped")
        continue

    print(f"\n--- {variant} model test ---")
    print(f"Endpoint: {endpoint}")

    headers = {}
    token = os.environ.get("SERVING_TOKEN", "")
    if token:
        headers["Authorization"] = f"Bearer {token}"

    ca_bundle = os.environ.get("SERVING_CA_BUNDLE", "")
    verify = ca_bundle if ca_bundle else True

    # 1. Chat completion test
    print("\n  1️⃣ Chat completion test")
    try:
        resp = httpx.post(
            f"{endpoint}/chat/completions",
            json={
                "model": endpoints_config.get("endpoints", {}).get(variant, {}).get("model_name", variant),
                "messages": [
                    {"role": "system", "content": "You are a banking support assistant."},
                    {"role": "user", "content": "What is the maximum daily transfer limit for premium accounts?"},
                ],
                "max_tokens": 256,
                "temperature": 0.1,
            },
            headers=headers,
            verify=verify,
            timeout=60,
        )
        resp.raise_for_status()
        result = resp.json()
        content = result["choices"][0]["message"]["content"]
        print(f"  ✅ Response: {content[:200]}")
    except Exception as exc:
        print(f"  ❌ Failed: {exc}")

    # 2. Tool call test
    print("\n  2️⃣ Tool call test")
    try:
        resp = httpx.post(
            f"{endpoint}/chat/completions",
            json={
                "model": endpoints_config.get("endpoints", {}).get(variant, {}).get("model_name", variant),
                "messages": [
                    {"role": "system", "content": "You are a banking assistant. Use tools when needed."},
                    {"role": "user", "content": "Please check the balance of account ACC-12345."},
                ],
                "tools": [{
                    "type": "function",
                    "function": {
                        "name": "get_account_balance",
                        "description": "Get the balance of a bank account",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "account_id": {"type": "string", "description": "Account ID"}
                            },
                            "required": ["account_id"],
                        },
                    },
                }],
                "max_tokens": 256,
                "temperature": 0.1,
            },
            headers=headers,
            verify=verify,
            timeout=60,
        )
        resp.raise_for_status()
        result = resp.json()
        msg = result["choices"][0]["message"]
        if msg.get("tool_calls"):
            tc = msg["tool_calls"][0]
            print(f"  ✅ Tool call: {tc['function']['name']}({tc['function']['arguments']})")
        else:
            print(f"  ⚠️  No tool call. Response: {msg.get('content', '')[:200]}")
    except Exception as exc:
        print(f"  ❌ Failed: {exc}")

print("\n" + "=" * 70)
print("Deployment complete! Next steps:")
print("  📓 06_rag_harness.ipynb — Building RAG harness")
print("  📓 07_evaluate.ipynb — Run evaluation")